# Notebook 01: BPE Tokenizer from Scratch

## Overview

- **Duration**: ~3 hours
- **Prerequisites**: Basic Python, understanding of text encoding
- **Learning Objectives**:
  1. Understand why subword tokenization is necessary for LLMs
  2. Implement the BPE training algorithm from scratch
  3. Implement encode/decode functions
  4. Compare your tokenizer with production tokenizers (tiktoken)

## Introduction

Tokenization is the first step in any NLP pipeline. It converts raw text into a sequence of integers that can be processed by neural networks.

### Why not just use characters or words?

- **Character-level**: Very long sequences, limited semantic meaning per token
- **Word-level**: Huge vocabulary, can't handle unknown words, no morphological awareness

### Byte Pair Encoding (BPE)

BPE is a **subword tokenization** algorithm that:
1. Starts with a vocabulary of individual bytes (256 tokens)
2. Iteratively merges the most frequent adjacent pairs
3. Builds up a vocabulary of common subwords

This gives us:
- A bounded vocabulary size
- Ability to encode any text (byte-level fallback)
- Efficient representation of common patterns

### Reference Materials

- [Sebastian Raschka's BPE Tutorial](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)
- [Hugging Face Tokenizers Course](https://huggingface.co/learn/llm-course/en/chapter6/5)
- [Karpathy's minbpe](https://github.com/karpathy/minbpe)

## 1. Setup

In [15]:
import sys
sys.path.insert(0, '../..')  # Add project root to path

# We'll only use standard library for most of this notebook
from collections import Counter
import json
from pathlib import Path

In [16]:
# Sample text for initial experiments
sample_text = """The quick brown fox jumps over the lazy dog.
The dog was not amused by the fox's antics.
Foxes are known for their cunning nature."""

print(f"Sample text ({len(sample_text)} characters):")
print(sample_text)

Sample text (130 characters):
The quick brown fox jumps over the lazy dog.
The dog was not amused by the fox's antics.
Foxes are known for their cunning nature.


## 2. Understanding Bytes and Unicode

Before implementing BPE, we need to understand how text is represented as bytes.

- **UTF-8** is the most common text encoding
- ASCII characters (a-z, A-Z, 0-9, punctuation) are 1 byte
- Other characters (ä, ü, 日, 本) can be 2-4 bytes

In [17]:
# Explore UTF-8 encoding
examples = ["hello", "Hëllo", "日本語", "👋🌍"]

for text in examples:
    encoded = text.encode("utf-8")
    print(f"'{text}' ({len(text)} chars) -> {len(encoded)} bytes: {list(encoded)}")

'hello' (5 chars) -> 5 bytes: [104, 101, 108, 108, 111]
'Hëllo' (5 chars) -> 6 bytes: [72, 195, 171, 108, 108, 111]
'日本語' (3 chars) -> 9 bytes: [230, 151, 165, 230, 156, 172, 232, 170, 158]
'👋🌍' (2 chars) -> 8 bytes: [240, 159, 145, 139, 240, 159, 140, 141]


In [18]:
# The base vocabulary: all 256 possible byte values
base_vocab = {i: bytes([i]) for i in range(256)} # for > 256, we need char(i)

# Show some examples
print("Some byte values and their meanings:")
# Iterate through selected byte values to demonstrate UTF-8 encoding
# 65=A, 97=a, 32=space, 10=newline, 195 and 228 are incomplete multi-byte sequences
for i in [65, 97, 32, 10, 195, 228]:
    # Create a bytes object from a single byte value
    byte_val = bytes([i])
    try:
        # Attempt to decode the byte as UTF-8
        decoded = byte_val.decode("utf-8")
        # Print: byte value (right-aligned in 3 chars) -> bytes representation -> decoded character
        print(f"  {i:3d} -> {byte_val} -> '{decoded}'")
    except:
        # If decoding fails, it means this byte is part of a multi-byte UTF-8 character
        # and cannot stand alone (e.g., 195 is the start of 'ä' which needs 2 bytes)
        print(f"  {i:3d} -> {byte_val} -> (partial UTF-8 sequence)")

Some byte values and their meanings:
   65 -> b'A' -> 'A'
   97 -> b'a' -> 'a'
   32 -> b' ' -> ' '
   10 -> b'\n' -> '
'
  195 -> b'\xc3' -> (partial UTF-8 sequence)
  228 -> b'\xe4' -> (partial UTF-8 sequence)


## 3. Exercises

### Exercise 3.1: Compute Pair Frequencies (20 min)

Implement a function `get_stats` that counts how often each adjacent pair of tokens appears.

For example, in the sequence `[1, 2, 3, 1, 2]`:
- Pair (1, 2) appears 2 times
- Pair (2, 3) appears 1 time
- Pair (3, 1) appears 1 time

In [19]:
####solution for 3.1
#O(n) time complexity, where n is the number of tokens in the input list. We iterate through the list once to count the pairs, and each pair is processed in constant time.
def get_stats(tokens: list) -> dict:
    """Count frequency of each symbol in the tokens list."""
    pairs = {}
    for i in range(len(tokens)-1):
        pair = (tokens[i], tokens[i+1])
        if pair not in pairs:
            pairs[pair] = 0
        pairs[pair] += 1
    
    return pairs

In [20]:
####test cell
# Test your implementation
test_ids = [1, 2, 3, 1, 2]
stats = get_stats(test_ids)
print(f"Stats for {test_ids}:")
print(stats)

# Validation
assert stats[(1, 2)] == 2, f"Expected (1,2) count to be 2, got {stats[(1, 2)]}"
assert stats[(2, 3)] == 1, f"Expected (2,3) count to be 1, got {stats[(2, 3)]}"
assert stats[(3, 1)] == 1, f"Expected (3,1) count to be 1, got {stats[(3, 1)]}"
print("✓ Exercise 3.1 passed!")

Stats for [1, 2, 3, 1, 2]:
{(1, 2): 2, (2, 3): 1, (3, 1): 1}
✓ Exercise 3.1 passed!


### Exercise 3.2: Merge Operation (30 min)

Implement the merge operation: replace all occurrences of a pair with a new token ID by defining a function `merge(ids: list[int], pair: tuple[int, int], new_id: int)`.

For example, if we merge pair (1, 2) into new token 256:
- `[1, 2, 3, 1, 2]` becomes `[256, 3, 256]`

In [21]:
####solution for 3.2
def merge(ids: list[int], pair: tuple[int, int], new_id: int):
    l = [i for i in ids]
    i = 0
    while i < len(l)-1:
        #print(f"Checking pair: ({l[i]}, {l[i+1]}) against {pair}")
        if l[i]==pair[0] and l[i+1]==pair[1]:
            l[i] = new_id
            l.pop(i+1)
        i += 1
    return l

In [22]:
####test cell
# Test your implementation
test_ids = [1, 2, 3, 1, 2]
merged = merge(test_ids, (1, 2), 256)
print(f"Merging (1, 2) -> 256 in {test_ids}:")
print(f"Result: {merged}")

# Validation
assert merged == [256, 3, 256], f"Expected [256, 3, 256], got {merged}"

# Edge case: overlapping patterns
test_ids2 = [1, 1, 1, 1]
merged2 = merge(test_ids2, (1, 1), 256)
print(f"Merging (1, 1) -> 256 in {test_ids2}: {merged2}")
assert merged2 == [256, 256], f"Expected [256, 256], got {merged2}"

print("✓ Exercise 3.2 passed!")

Merging (1, 2) -> 256 in [1, 2, 3, 1, 2]:
Result: [256, 3, 256]
Merging (1, 1) -> 256 in [1, 1, 1, 1]: [256, 256]
✓ Exercise 3.2 passed!


### Exercise 3.3: BPE Training (45 min)

Now define a class named `BasicTokenizer` to implement the full BPE training algorithm. The class includes three functions:

1. the training function can convert text to bytes (initial tokens), and repeat until we reach target vocab size:
   - Find the most frequent pair
   - Create a new token for that pair
   - Replace all occurrences of the pair with the new token
   - Record the merge rule
2. An `encode` function that encode the text to token ids.
3. A `decode` function that decode the token ids to text.

In [23]:
####solution for 3.3 (from 11:30 to 22:07)
from tqdm import tqdm
import time

class BasicTokenizer:
    def __init__(self):
        self.encoded = [] # training text to token ids
        self.merges = {} # merge rule
        self.vocab = {} # vocabulary, 256 <= |vocab| <= vocab_size
        # self.reverse_vocab = {} # reverse mapping from vocab to id
        self.verbose = False
        self.vocab_size = 256

    def _print(self, *args, **kwargs):
        """Print only if verbose mode is enabled."""
        if self.verbose:
            print(*args, **kwargs)

    def train(self,sample_text: list[str], vocab_size: int, verbose: bool=True):
        self.verbose = verbose
        self.vocab_size = vocab_size
        self._print("Training tokenizer...")
        self._print("Set initial vocabulary...")
        self.vocab = {i: chr(i).encode('utf-8') for i in range(256)}
        self._print("Initial vocabulary", self.vocab)
        self._print("Encoding training text...")
        self.encoded = []
        with tqdm(total=len(sample_text), desc="initial UTF-8 encoding of text lines", unit="encode", disable=not self.verbose) as pbar:
            for i, text in enumerate(sample_text):
                self.encoded.extend(list(text.encode("utf-8", errors='replace')))
                pbar.update(1)

        self._print("Initial encoded tokens", len(self.encoded))
        if self.verbose:
                for token_id, token_bytes in list(self.vocab.items())[-10:]:
                    self._print(f"  ID {token_id}: {token_bytes} → '{token_bytes.decode('utf-8', errors='replace')}'")
        
        self.merging()
        if self.vocab_size > 256:
            self._print("Final vocabulary", len(self.vocab))
            print("Last 10 tokens in vocabulary:")
            if self.verbose:
                for token_id, token_bytes in list(self.vocab.items())[-10:]:
                    self._print(f"  ID {token_id}: {token_bytes} → '{token_bytes.decode('utf-8', errors='replace')}'")

        # Reverse the vocabulary merges for decoding (not needed for encoding since we can directly look up the vocab)
        # self.reverse_merges = {v: k for k, v in reversed(self.merges.items())}
        # self._print("Reverse merges", self.reverse_merges)
        


    """Merging function only for the training"""
    def merging(self):
        self._print("Calculating pair frequencies...")
        stats = get_stats(self.encoded)
        # Maximum of pairs in O(n)
        # key: the token pair that occurs most often
        max_pair_key = max(stats, key=stats.get)
        # value: the amount that the pais occurs 
        max_pair_value = max(stats.values())
        #copy
        total_merges = self.vocab_size - 256
        self._print("Merge to convergence...")
        # Create progress bar
        with tqdm(total=total_merges, desc="Merging pairs", unit=" merge", disable=not self.verbose) as pbar:
            # This loop will keep merging the most frequent pair until no pair occurs more than once
            # uses get_stats to find the most frequent pair (max value in dict) and merge to update the token list
            while max_pair_value > 1 and len(self.vocab) < self.vocab_size: # limit vocab size to 65536
                # the newest token to encode has a new id of exactly one more than the current max token id in the vocab
                new_token_key = max(self.vocab)+1
                # add the token to the vocab
                self.vocab[new_token_key] = (str(self.vocab[max_pair_key[0]].decode('utf-8')) + str(self.vocab[max_pair_key[1]].decode('utf-8'))).encode('utf-8')
                #self._print(new_token_key, str(self.vocab[max_pair_key[0]]) + str(self.vocab[max_pair_key[1]]))

                # merge on the pair (max_key) that occurs most often
                self.encoded = merge(self.encoded, max_pair_key, new_token_key)
            
                # new merge rule for the most frequent pair
                self.merges[max_pair_key] = int(new_token_key)
            
                # stats of new merge?
                stats = get_stats(self.encoded)
                max_pair_key = max(stats, key=stats.get)
                max_pair_value = max(stats.values())
            
                
                pbar.update(1)
                pbar.set_postfix({
                'vocab_size': len(self.vocab),
                'freq': max_pair_value,
                'pair': str(max_pair_key)[:20]  # Truncate for display
                })

    """encoding a new sample text"""
    def encode(self, sample_text: str, verbose: bool=True):
        if not self.vocab:
            print("Tokenizer must be trained before encoding.")
            return []
        self.verbose = verbose
        self._print("Encoding sample text...")
        encode_test = list(sample_text.encode("utf-8", errors='replace'))
        self._print("Encode-Merge...")
        total_merges = self.vocab_size - 256
        # Create progress bar
        with tqdm(total=total_merges, desc="Merging pairs", unit=" merge", disable=not self.verbose) as pbar:
            for i, (pair, new_id) in enumerate(self.merges.items()):
                encode_test = merge(ids=encode_test, pair=pair, new_id=new_id)
            
                pbar.update(1)
        
        return encode_test

    def decode(self, encoding: list, verbose: bool=True):
        if not self.merges:
            print("Tokenizer must be trained before decoding.")
            return []
    
        
        # decode directly by looking up each enconding id in vocab
        decoding = ''.join([self.vocab[id].decode('utf-8') for id in encoding])
        
        return decoding
        return self.reverse_vocab

tk = BasicTokenizer()
print(sample_text)
tk.encode(sample_text)
tk.train([sample_text], 280, True)
print("Vocabulary:", tk.vocab)
print("Encoded tokens:",tk.encoded)
print("Merged Tokens", tk.merges)

The quick brown fox jumps over the lazy dog.
The dog was not amused by the fox's antics.
Foxes are known for their cunning nature.
Tokenizer must be trained before encoding.
Training tokenizer...
Set initial vocabulary...
Initial vocabulary {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C'

initial UTF-8 encoding of text lines: 100%|██████████| 1/1 [00:00<?, ?encode/s]


Initial encoded tokens 130
  ID 246: b'\xc3\xb6' → 'ö'
  ID 247: b'\xc3\xb7' → '÷'
  ID 248: b'\xc3\xb8' → 'ø'
  ID 249: b'\xc3\xb9' → 'ù'
  ID 250: b'\xc3\xba' → 'ú'
  ID 251: b'\xc3\xbb' → 'û'
  ID 252: b'\xc3\xbc' → 'ü'
  ID 253: b'\xc3\xbd' → 'ý'
  ID 254: b'\xc3\xbe' → 'þ'
  ID 255: b'\xc3\xbf' → 'ÿ'
Calculating pair frequencies...
Merge to convergence...


Merging pairs:  79%|███████▉  | 19/24 [00:00<00:00, 181.74 merge/s, vocab_size=275, freq=1, pair=(261, 113)]

Final vocabulary 275
Last 10 tokens in vocabulary:
  ID 265: b'own' → 'own'
  ID 266: b'own ' → 'own '
  ID 267: b'own fo' → 'own fo'
  ID 268: b'r t' → 'r t'
  ID 269: b'y ' → 'y '
  ID 270: b'do' → 'do'
  ID 271: b'dog' → 'dog'
  ID 272: b'.\n' → '.
'
  ID 273: b's a' → 's a'
  ID 274: b're' → 're'
Vocabulary: {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b

In [24]:
encoded_result = tk.encode(sample_text=sample_text)

print(encoded_result)
# Exact match (order and values):
if encoded_result == tk.encoded:
    print("✓ Encoding matches training!")
else:
    print("✗ Mismatch!")
    print(f"Expected: {tk.encoded}")
    print(f"Got:      {encoded_result}")

decoded_result = tk.decode(encoded_result)

print(decoded_result)
# Exact match (order and values):
if decoded_result == sample_text:
    print("✓ Decoding matches sample text!")
else:
    print("✗ Mismatch!")
    print(f"Expected: {sample_text}")
    print(f"Got:      {decoded_result}")

Encoding sample text...
Encode-Merge...


Merging pairs:  79%|███████▉  | 19/24 [00:00<00:00, 14202.78 merge/s]

[261, 113, 117, 262, 107, 263, 114, 267, 120, 32, 106, 117, 109, 112, 258, 111, 118, 101, 268, 257, 108, 97, 122, 269, 271, 272, 261, 271, 32, 119, 97, 258, 110, 111, 116, 32, 97, 109, 117, 115, 101, 100, 263, 269, 116, 257, 259, 120, 39, 273, 110, 116, 262, 115, 272, 70, 111, 120, 101, 273, 274, 32, 107, 110, 267, 268, 256, 105, 260, 99, 117, 110, 110, 105, 110, 103, 32, 110, 97, 116, 117, 274, 46]
✓ Encoding matches training!
The quick brown fox jumps over the lazy dog.
The dog was not amused by the fox's antics.
Foxes are known for their cunning nature.
✓ Decoding matches sample text!


In [25]:
####test cell
# Test your implementation on the sample text
tokenizer = BasicTokenizer()
tokenizer.train(sample_text, vocab_size=280, verbose=True)

print(f"\nLearned {len(tokenizer.merges)} merges")
print(f"Vocabulary size: {len(tokenizer.vocab)}")

# Show some learned merges
print("\nFirst 10 merges:")
for i, ((a, b), new_id) in enumerate(tokenizer.merges.items()):
    if i >= 10:
        break
    token_a = tokenizer.vocab[a].decode('utf-8', errors='replace')
    token_b = tokenizer.vocab[b].decode('utf-8', errors='replace')
    merged = tokenizer.vocab[new_id].decode('utf-8', errors='replace')
    print(f"  '{token_a}' + '{token_b}' -> '{merged}' (id={new_id})")


Training tokenizer...
Set initial vocabulary...
Initial vocabulary {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85

initial UTF-8 encoding of text lines: 100%|██████████| 130/130 [00:00<00:00, 228524.53encode/s]


Initial encoded tokens 130
  ID 246: b'\xc3\xb6' → 'ö'
  ID 247: b'\xc3\xb7' → '÷'
  ID 248: b'\xc3\xb8' → 'ø'
  ID 249: b'\xc3\xb9' → 'ù'
  ID 250: b'\xc3\xba' → 'ú'
  ID 251: b'\xc3\xbb' → 'û'
  ID 252: b'\xc3\xbc' → 'ü'
  ID 253: b'\xc3\xbd' → 'ý'
  ID 254: b'\xc3\xbe' → 'þ'
  ID 255: b'\xc3\xbf' → 'ÿ'
Calculating pair frequencies...
Merge to convergence...


Merging pairs:  79%|███████▉  | 19/24 [00:00<00:00, 564.07 merge/s, vocab_size=275, freq=1, pair=(261, 113)]


Final vocabulary 275
Last 10 tokens in vocabulary:
  ID 265: b'own' → 'own'
  ID 266: b'own ' → 'own '
  ID 267: b'own fo' → 'own fo'
  ID 268: b'r t' → 'r t'
  ID 269: b'y ' → 'y '
  ID 270: b'do' → 'do'
  ID 271: b'dog' → 'dog'
  ID 272: b'.\n' → '.
'
  ID 273: b's a' → 's a'
  ID 274: b're' → 're'

Learned 19 merges
Vocabulary size: 275

First 10 merges:
  'h' + 'e' -> 'he' (id=256)
  'he' + ' ' -> 'he ' (id=257)
  's' + ' ' -> 's ' (id=258)
  'f' + 'o' -> 'fo' (id=259)
  'r' + ' ' -> 'r ' (id=260)
  'T' + 'he ' -> 'The ' (id=261)
  'i' + 'c' -> 'ic' (id=262)
  ' ' + 'b' -> ' b' (id=263)
  'o' + 'w' -> 'ow' (id=264)
  'ow' + 'n' -> 'own' (id=265)


In [26]:
####test cell
# Test encode/decode
test_text = "The fox jumps!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)

print(f"Original: '{test_text}'")
print(f"Encoded:  {encoded}")
print(f"Decoded:  '{decoded}'")

# Validation: round-trip should preserve text
assert decoded == test_text, f"Round-trip failed: '{decoded}' != '{test_text}'"
print("\n✓ Round-trip test passed!")

Encoding sample text...
Encode-Merge...


Merging pairs:  79%|███████▉  | 19/24 [00:00<?, ? merge/s]

Original: 'The fox jumps!'
Encoded:  [261, 259, 120, 32, 106, 117, 109, 112, 115, 33]
Decoded:  'The fox jumps!'

✓ Round-trip test passed!


### Exercise 3.4: Train on Real Data (30 min)

Now let's train a tokenizer on a larger dataset: a sample of TinyStories.

- Download a text sample of TinyStories. Hint: use `load_dataset("roneneldan/TinyStories", split="train[:5000]")`.
- Train a larger tokenizer using the text sample downloaded.

In [27]:
####solution 3.4
from datasets import load_dataset

# Download a text sample of TinyStories
dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")
print(dataset[0])  # Show the first example to verify it loaded correctly

# Extract text from the dataset
training_text = [example["text"] for example in dataset]

print(f"Downloaded {len(dataset)} stories")
print(f"Total characters: {len("".join(training_text)):,}")

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}
Downloaded 5000 stories
Total characters: 4,176,740


In [ ]:
# Train a larger tokenizer
# This takes ~90min on a CPU with vocab size of 2000.
# But later merges are ~1sec and early merges are serveral minutes,
# so maybe larger vocabs are possible with more time or a GPU.
tokenizer_large = BasicTokenizer()
tokenizer_large.train(training_text, vocab_size=2000, verbose=True)

Training tokenizer...
Set initial vocabulary...
Initial vocabulary {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85

initial UTF-8 encoding of text lines: 100%|██████████| 5000/5000 [00:00<00:00, 15060.37encode/s]


Initial encoded tokens 4187330
  ID 246: b'\xc3\xb6' → 'ö'
  ID 247: b'\xc3\xb7' → '÷'
  ID 248: b'\xc3\xb8' → 'ø'
  ID 249: b'\xc3\xb9' → 'ù'
  ID 250: b'\xc3\xba' → 'ú'
  ID 251: b'\xc3\xbb' → 'û'
  ID 252: b'\xc3\xbc' → 'ü'
  ID 253: b'\xc3\xbd' → 'ý'
  ID 254: b'\xc3\xbe' → 'þ'
  ID 255: b'\xc3\xbf' → 'ÿ'
Calculating pair frequencies...
Merge to convergence...


Merging pairs: 100%|██████████| 1744/1744 [1:32:35<00:00,  3.19s/ merge, vocab_size=2000, freq=166, pair=(1146, 362)] 

Final vocabulary 2000
Last 10 tokens in vocabulary:
  ID 1990: b'warm' → 'warm'
  ID 1991: b'Thank you, ' → 'Thank you, '
  ID 1992: b', and the ' → ', and the '
  ID 1993: b'ly and ' → 'ly and '
  ID 1994: b'catch ' → 'catch '
  ID 1995: b'yummy ' → 'yummy '
  ID 1996: b'was so happy' → 'was so happy'
  ID 1997: b'was not ' → 'was not '
  ID 1998: b'him a ' → 'him a '
  ID 1999: b'fla' → 'fla'


In [32]:
"""Do not run this, as it will overwrite the trained tokenizer from the previous exercises and we want to keep that for later use."""
# pickle the trained tokenizer data for later use
import pickle

# Save only the essential trained components
tokenizer_data = {
    'vocab': tokenizer_large.vocab,
    'merges': tokenizer_large.merges,
    'vocab_size': tokenizer_large.vocab_size
}

with open("models/trained_tokenizer_data.pkl", "wb") as f:
    pickle.dump(tokenizer_data, f)

print("Tokenizer data saved!")


Tokenizer data saved!


In [33]:
import pickle

# Load the data
with open("models/trained_tokenizer_data.pkl", "rb") as f:
    tokenizer_data = pickle.load(f)

# Create a new tokenizer and restore the trained data
tokenizer_large= BasicTokenizer()
tokenizer_large.vocab = tokenizer_data['vocab']
tokenizer_large.merges = tokenizer_data['merges']
tokenizer_large.vocab_size = tokenizer_data['vocab_size']

print("Tokenizer loaded!")

Tokenizer loaded!


In [34]:
####test cell
# Analyze tokenization quality
test_texts = [
    "Once upon a time, there was a little girl.",
    "The quick brown fox jumps over the lazy dog.",
    "She loved to play with her toys.",
]

print("Tokenization examples:\n")
for text in test_texts:
    tokens = tokenizer_large.encode(text)
    token_strs = [tokenizer_large.vocab[t].decode('utf-8', errors='replace') for t in tokens]
    
    print(f"Text: '{text}'")
    print(f"  {len(text)} chars -> {len(tokens)} tokens (ratio: {len(text)/len(tokens):.2f})")
    print(f"  Tokens: {token_strs[:15]}{'...' if len(tokens) > 15 else ''}")
    print()

Tokenization examples:

Encoding sample text...
Encode-Merge...


Merging pairs: 100%|██████████| 1744/1744 [00:00<00:00, 629452.39 merge/s]


Text: 'Once upon a time, there was a little girl.'
  42 chars -> 4 tokens (ratio: 10.50)
  Tokens: ['Once upon a tim', 'e, there was a ', 'little girl', '.']

Encoding sample text...
Encode-Merge...


Merging pairs: 100%|██████████| 1744/1744 [00:00<00:00, 238385.73 merge/s]


Text: 'The quick brown fox jumps over the lazy dog.'
  44 chars -> 17 tokens (ratio: 2.59)
  Tokens: ['The ', 'qu', 'ic', 'k ', 'bro', 'wn ', 'fo', 'x ', 'jump', 's ', 'over ', 'the ', 'la', 'z', 'y ']...

Encoding sample text...
Encode-Merge...


Merging pairs: 100%|██████████| 1744/1744 [00:00<00:00, 394971.18 merge/s]

Text: 'She loved to play with her toys.'
  32 chars -> 7 tokens (ratio: 4.57)
  Tokens: ['She ', 'loved to ', 'play with ', 'her ', 'toy', 's', '.']



## 4. Compare with tiktoken

Let's compare our tokenizer with OpenAI's `tiktoken` (used by GPT-4).

In [35]:
#%pip install tiktoken #install tiktoken if you don't have it
import tiktoken

# Load GPT-4's tokenizer
enc = tiktoken.get_encoding("cl100k_base")

print(f"tiktoken vocab size: {enc.n_vocab:,}")

# Compare tokenizations
test_text = "Hello, world! This is a test of tokenization. 日本語も試してみましょう。"

our_tokens = tokenizer_large.encode(test_text)
tiktoken_tokens = enc.encode(test_text)

print(f"\nTest text: '{test_text}'")
print(f"\nOur tokenizer: {len(our_tokens)} tokens")
print(f"tiktoken:      {len(tiktoken_tokens)} tokens")

print(f"\nOur tokens:     {our_tokens[:20]}...")
print(f"tiktoken tokens: {tiktoken_tokens[:20]}...")

tiktoken vocab size: 100,277
Encoding sample text...
Encode-Merge...


Merging pairs: 100%|██████████| 1744/1744 [00:00<00:00, 162350.55 merge/s]


Test text: 'Hello, world! This is a test of tokenization. 日本語も試してみましょう。'

Our tokenizer: 56 tokens
tiktoken:      27 tokens

Our tokens:     [1683, 298, 270, 1926, 425, 1613, 1504, 116, 414, 1162, 320, 107, 276, 1145, 1092, 281, 262, 230, 151, 165]...
tiktoken tokens: [9906, 11, 1917, 0, 1115, 374, 264, 1296, 315, 4037, 2065, 13, 76502, 22656, 45918, 252, 32977, 50520, 99, 39926]...


In [36]:
# Visualize how each tokenizer splits the text
def visualize_tokens(tokenizer, text, name):
    if hasattr(tokenizer, 'encode'):
        if hasattr(tokenizer, 'vocab'):
            # Our tokenizer
            tokens = tokenizer.encode(text)
            token_strs = [tokenizer.vocab[t].decode('utf-8', errors='replace') for t in tokens]
        else:
            # tiktoken
            tokens = tokenizer.encode(text)
            token_strs = [tokenizer.decode([t]) for t in tokens]
    
    print(f"\n{name}:")
    print(f"  Tokens: {len(tokens)}")
    print(f"  Split: |{'|'.join(repr(s)[1:-1] for s in token_strs)}|")

test = "The dog was running quickly."
visualize_tokens(tokenizer_large, test, "Our BPE")
visualize_tokens(enc, test, "tiktoken (GPT-4)")

Encoding sample text...
Encode-Merge...


Merging pairs: 100%|██████████| 1744/1744 [00:00<00:00, 441079.73 merge/s]


Our BPE:
  Tokens: 7
  Split: |The |dog |was |running |quick|ly|.|

tiktoken (GPT-4):
  Tokens: 6
  Split: |The| dog| was| running| quickly|.|


## 5. Challenge Exercise: Regex Pre-tokenization (Optional)

GPT-4 uses regex pre-tokenization to split text before applying BPE.
This prevents merges across word boundaries.

The pattern is:
```python
GPT4_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```

Can you modify your tokenizer to use this pattern?

In [1]:
# Hint: You'll need the 'regex' module (not 're') for Unicode support
# uv pip install regex

import regex as re

GPT4_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

# Test the pattern
test_text = "Hello, world! I've been testing this. Numbers: 12345"
chunks = re.findall(GPT4_PATTERN, test_text)
print(f"Pre-tokenized chunks: {chunks}")

Pre-tokenized chunks: ['Hello', ',', ' world', '!', ' I', "'ve", ' been', ' testing', ' this', '.', ' Numbers', ':', ' ', '123', '45']


## 6. Summary

### What You Learned

1. **Why BPE**: Balances vocabulary size vs sequence length
2. **The Algorithm**: Iteratively merge most frequent pairs
3. **Implementation**: `get_stats()`, `merge()`, training loop
4. **Encoding/Decoding**: Apply merges in order during encoding

### Key Takeaways

- Tokenization is crucial for LLM performance
- BPE adapts to the training corpus
- Larger vocabulary = shorter sequences but more embeddings to learn
- Production tokenizers add regex pre-tokenization for better boundaries

### Preparation for Next Notebook

In Notebook 02, we'll implement **token embeddings** and **positional encodings** - the first learnable components of the transformer.

In [ ]:
# Save your trained tokenizer for later notebooks
import pickle

save_path = Path("model/tokenizer.pkl")
with open(save_path, "wb") as f:
    pickle.dump(tokenizer_large, f)

print(f"Tokenizer saved to {save_path}")

Tokenizer saved to tokenizer.pkl
